<a href="https://colab.research.google.com/github/dilshanirubasinghe/Large-Scale-Imagery-Splitter/blob/main/image_clip_256_256.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 108.6 MB/s eta 0:00:00


In [ ]:
import rasterio
import os


In [ ]:
import rasterio
import os
from math import ceil

def create_tiles(image_path, output_folder, tile_size=512):
    # Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Open the source image
    with rasterio.open(image_path) as src:
        # Get image dimensions
        width = src.width
        height = src.height

        # Calculate number of tiles using ceiling division
        num_tiles_x = ceil(width / tile_size)
        num_tiles_y = ceil(height / tile_size)

        # Tile creation loop
        for i in range(num_tiles_x):
            for j in range(num_tiles_y):
                # Calculate window coordinates
                x_off = i * tile_size
                y_off = j * tile_size

                # Determine actual tile dimensions
                win_width = min(tile_size, width - x_off)
                win_height = min(tile_size, height - y_off)

                # Create window
                window = rasterio.windows.Window(x_off, y_off, win_width, win_height)

                # Read window data
                tile_data = src.read(window=window)

                # Create output filename
                output_file = os.path.join(output_folder, f'tile_{i}_{j}.tif')

                # Calculate transform for the tile
                transform = rasterio.windows.transform(window, src.transform)

                # Update metadata
                meta = src.meta.copy()
                meta.update({
                    'height': win_height,
                    'width': win_width,
                    'transform': transform,
                    'driver': 'GTiff'
                })

                # Write tile
                with rasterio.open(output_file, 'w', **meta) as dst:
                    dst.write(tile_data)

    print(f"Successfully created {num_tiles_x * num_tiles_y} tiles in {output_folder}")

# Colab paths
image_path = '/content/drive/MyDrive/dronenew/66c6d07c0378ba0001bb5e53.tif'
output_folder = '/content/drive/MyDrive/dronenew/output'

# Generate tiles
create_tiles(image_path, output_folder)

Successfully created 308 tiles in /content/drive/MyDrive/dronenew/output
